In [1]:
!pip install ultralytics tensorflow opencv-python matplotlib seaborn scikit-learn streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 50.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_PATH = "/content/drive/MyDrive/Road/data"

IMAGES_PATH = BASE_PATH + "/images"
LABELS_PATH = BASE_PATH + "/labels"

print("Images:", len(os.listdir(IMAGES_PATH)))
print("Labels:", len(os.listdir(LABELS_PATH)))

Images: 2009
Labels: 2009


In [4]:
import os

dataset = "/content/dataset"

for split in ["train","val","test"]:

    os.makedirs(f"{dataset}/{split}/images",exist_ok=True)
    os.makedirs(f"{dataset}/{split}/labels",exist_ok=True)

In [5]:
import random

images = os.listdir(IMAGES_PATH)
random.shuffle(images)

train_split = int(0.7 * len(images))
val_split = int(0.9 * len(images))

train_files = images[:train_split]
val_files = images[train_split:val_split]
test_files = images[val_split:]

In [6]:
import shutil

def move_files(files, split):

    for img in files:

        src_img = f"{IMAGES_PATH}/{img}"
        dst_img = f"/content/dataset/{split}/images/{img}"

        shutil.copy(src_img,dst_img)

        label = img.replace(".jpg",".txt")

        src_lbl = f"{LABELS_PATH}/{label}"
        dst_lbl = f"/content/dataset/{split}/labels/{label}"

        if os.path.exists(src_lbl):
            shutil.copy(src_lbl,dst_lbl)

move_files(train_files,"train")
move_files(val_files,"val")
move_files(test_files,"test")

print("Dataset Split Completed")

Dataset Split Completed


In [7]:
import os

print("Train images:", len(os.listdir("/content/dataset/train/images")))
print("Val images:", len(os.listdir("/content/dataset/val/images")))
print("Test images:", len(os.listdir("/content/dataset/test/images")))

Train images: 1406
Val images: 402
Test images: 201


In [8]:
%%writefile /content/dataset/data.yaml

path: /content/dataset

train: train/images
val: val/images
test: test/images

names:
  0: crack
  1: pothole
  2: manhole
  3: normal

Writing /content/dataset/data.yaml


In [9]:
!cat /content/dataset/data.yaml


path: /content/dataset

train: train/images
val: val/images
test: test/images

names:
  0: crack
  1: pothole
  2: manhole
  3: normal


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=